In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


SOURCE_DIR = Path("../data/csv")

OUTPUT_PATH = Path("../data/raw/btc_network_daily.csv")

START_DATE = "2020-01-01"
END_DATE = "2026-06-30"


def load_blockchain_csv(filename: str, value_column: str) -> pd.DataFrame:
    """
    Загружает CSV формата:
    2009-01-17 00:00:00,109.0
    """
    path = SOURCE_DIR / filename

    if not path.exists():
        raise FileNotFoundError(f"Файл не найден: {path}")

    df = pd.read_csv(
        path,
        header=None,
        names=["date", value_column]
    )

    df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.floor("D")
    df[value_column] = pd.to_numeric(df[value_column], errors="coerce")

    df = df.dropna(subset=["date", value_column])
    df = df.groupby("date", as_index=False)[value_column].mean()
    df = df.sort_values("date").reset_index(drop=True)

    return df


# 1. Количество уникальных адресов
unique_addresses = load_blockchain_csv(
    "n-unique-addresses.csv",
    "unique_addresses"
)

# 2. Оценочный объем транзакций BTC
transfer_volume = load_blockchain_csv(
    "estimated-transaction-volume.csv",
    "transfer_volume_btc"
)

# 3. Суммарные комиссии за день в USD
transaction_fees_usd = load_blockchain_csv(
    "transaction-fees-usd.csv",
    "transaction_fees_usd"
)

# 4. Количество транзакций за день
n_transactions = load_blockchain_csv(
    "n-transactions.csv",
    "n_transactions"
)


network = (
    unique_addresses
    .merge(transfer_volume, on="date", how="inner")
    .merge(transaction_fees_usd, on="date", how="inner")
    .merge(n_transactions, on="date", how="inner")
)


network["avg_fee_usd"] = np.where(
    network["n_transactions"] > 0,
    network["transaction_fees_usd"] / network["n_transactions"],
    0.0
)


network = network[
    (network["date"] >= pd.to_datetime(START_DATE)) &
    (network["date"] <= pd.to_datetime(END_DATE))
].copy()


network = network[
    [
        "date",
        "unique_addresses",
        "transfer_volume_btc",
        "avg_fee_usd"
    ]
]


if network.empty:
    raise ValueError("После фильтрации по датам итоговый набор сетевых признаков пустой.")

if network.isna().any().any():
    raise ValueError(f"В итоговом наборе есть пропуски:\n{network.isna().sum()}")

if not np.isfinite(network[["unique_addresses", "transfer_volume_btc", "avg_fee_usd"]].to_numpy()).all():
    raise ValueError("В итоговом наборе есть бесконечные значения.")

if (network[["unique_addresses", "transfer_volume_btc", "avg_fee_usd"]] < 0).any().any():
    raise ValueError("В итоговом наборе есть отрицательные значения.")


network["date"] = network["date"].dt.strftime("%Y-%m-%d")


OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

network.to_csv(OUTPUT_PATH, index=False)

print(f"Файл создан: {OUTPUT_PATH}")
print(f"Количество строк: {len(network)}")
print()
print(network.head())
print()
print(network.tail())

Файл создан: data\raw\btc_network_daily.csv
Количество строк: 2358

            date  unique_addresses  transfer_volume_btc  avg_fee_usd
3388  2020-01-01          389706.0         38648.246791     0.285298
3389  2020-01-02          475461.0         98003.465802     0.330065
3390  2020-01-03          523231.0        236599.386904     0.375351
3391  2020-01-04          465366.0         85109.877765     0.317010
3392  2020-01-05          459065.0         47803.863974     0.307641

            date  unique_addresses  transfer_volume_btc  avg_fee_usd
5741  2026-06-25          522671.0        186630.020828     0.271334
5742  2026-06-26          521746.0        180313.117380     0.312712
5743  2026-06-27          452506.0         39050.509706     0.209717
5744  2026-06-28          416739.0         35504.363473     0.247882
5745  2026-06-29          490860.0        139093.596563     0.403671
